# Import Data From Source

In [16]:
import sys
import pandas as pd
import pickle
sys.path.append('../utils')

In [2]:
from data_utils import load_data
df = load_data()
df.head()

,Season,Episode,Character,Line
0,1,1,Boys,"School day, school day, teacher's golden ru...\n"
1,1,1,Kyle,"Ah, damn it! My little brother's trying to fol..."
2,1,1,Ike,Zeeponanner.\n
3,1,1,Kyle,"Ike, you can't come to school with me. \n"
4,1,1,Cartman,"Yeah, go home you little dildo.\n"


Only the last 2 columns interest us

In [3]:
df_dial = df.iloc[:,-2:]

In [4]:
df_dial.head()

,Character,Line
0,Boys,"School day, school day, teacher's golden ru...\n"
1,Kyle,"Ah, damn it! My little brother's trying to fol..."
2,Ike,Zeeponanner.\n
3,Kyle,"Ike, you can't come to school with me. \n"
4,Cartman,"Yeah, go home you little dildo.\n"


In [5]:
df_dial.info()

<class 'pandas.DataFrame'>
RangeIndex: 73139 entries, 0 to 73138
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Character  73139 non-null  str  
 1   Line       73139 non-null  str  
dtypes: str(2)
memory usage: 1.1 MB


We should delete the "\n" at the end of each line. Also trimming whitespace

In [6]:
df_dial["Line"] = df_dial["Line"].str.replace("\n", " ", regex=False).str.strip()

In [7]:
df_dial.head()

,Character,Line
0,Boys,"School day, school day, teacher's golden ru..."
1,Kyle,"Ah, damn it! My little brother's trying to fol..."
2,Ike,Zeeponanner.
3,Kyle,"Ike, you can't come to school with me."
4,Cartman,"Yeah, go home you little dildo."


# Data Processing

In [8]:
# Remove duplicates
df_clean = df_dial.drop_duplicates()
print(f"After removing duplicates: {len(df_clean):,} lines")


# Strip whitespace from character names
df_clean['Character'] = df_clean['Character'].str.strip()
df_clean['Line'] = df_clean['Line'].str.strip()

# Remove empty lines
df_clean = df_clean[df_clean['Line'].str.len() > 0]
print(f"After removing empty lines: {len(df_clean):,} lines")

After removing duplicates: 69,696 lines
After removing empty lines: 69,696 lines


## Remove infrequent characters

In [9]:
# Select top K characters by number of lines
character_counts = df_clean['Character'].value_counts()
top_characters = character_counts[character_counts > 200].index.tolist()  # Keep characters with more than 200 lines

print(f"\nTop {len(top_characters)} characters:")
for i, char in enumerate(top_characters, 1):
    count = character_counts[char]
    print(f"  {i:2d}. {char:20s} - {count:5d} lines ({count/len(df_clean)*100:.1f}%)")

# Filter dataset to only include top K characters
df_model = df_clean[df_clean['Character'].isin(top_characters)].copy()
print(f"\nFiltered dataset: {len(df_model):,} lines ({len(df_model)/len(df_clean)*100:.1f}% of total)")


Top 31 characters:
   1. Cartman              -  9518 lines (13.7%)
   2. Stan                 -  6978 lines (10.0%)
   3. Kyle                 -  6547 lines (9.4%)
   4. Butters              -  2581 lines (3.7%)
   5. Randy                -  2580 lines (3.7%)
   6. Mr. Garrison         -  1065 lines (1.5%)
   7. Sharon               -   867 lines (1.2%)
   8. Chef                 -   857 lines (1.2%)
   9. Kenny                -   751 lines (1.1%)
  10. Jimmy                -   677 lines (1.0%)
  11. Mr. Mackey           -   641 lines (0.9%)
  12. Gerald               -   637 lines (0.9%)
  13. Liane                -   578 lines (0.8%)
  14. Wendy                -   562 lines (0.8%)
  15. Jimbo                -   549 lines (0.8%)
  16. Sheila               -   547 lines (0.8%)
  17. Announcer            -   407 lines (0.6%)
  18. Stephen              -   371 lines (0.5%)
  19. Craig                -   347 lines (0.5%)
  20. Jesus                -   309 lines (0.4%)
  21. Clyde       

In [10]:
# Save to parquet
df_model.to_parquet("../01 - Inputs/south_park_dialogues.parquet", index=False)

# Separation between Train, Test and Validation

In [ ]:
from sklearn.model_selection import train_test_split
from data_utils import create_character_mapping
# We are aiming to determine who is talking based on the line

target = "Character"

# Separate features / target
X = df_model.drop(columns=[target])
y = df_model[target]

# Train (70%) / rest (30%) - stratified by character
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Validation (15%) / Test (15%) - stratified by character
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
# Ensure all classes are represented in all splits
assert len(y_train.value_counts()) == len(y_val.value_counts())
assert len(y_val.value_counts()) == len(y_test.value_counts())

# Create the integer mapping for the outcome variable and apply it to the splits
char_to_label, label_to_char = create_character_mapping(df_model, top_characters)
train_labels = y_train.map(char_to_label)
val_labels = y_val.map(char_to_label)
test_labels = y_test.map(char_to_label)


Character to label mapping:
  0: Cartman
  1: Stan
  2: Kyle
  3: Butters
  4: Randy
  5: Mr. Garrison
  6: Sharon
  7: Chef
  8: Kenny
  9: Jimmy
  10: Mr. Mackey
  11: Gerald
  12: Liane
  13: Wendy
  14: Jimbo
  15: Sheila
  16: Announcer
  17: Stephen
  18: Craig
  19: Jesus
  20: Clyde
  21: Principal Victoria
  22: Linda
  23: Token
  24: Mrs. Garrison
  25: Terrance
  26: Tweek
  27: Mayor
  28: Phillip
  29: Bebe
  30: Man


In [17]:
with open("../01 - Inputs/char_to_label.pkl", "wb") as f:
    pickle.dump(char_to_label, f)
with open("../01 - Inputs/label_to_char.pkl", "wb") as f:
    pickle.dump(label_to_char, f)

In [ ]:

train_data = pd.concat([X_train, y_train], axis=1)
val_data = pd.concat([X_val, y_val], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_parquet("../01 - Inputs/train_data.parquet", index=False)
val_data.to_parquet("../01 - Inputs/val_data.parquet", index=False)
test_data.to_parquet("../01 - Inputs/test_data.parquet", index=False)

train_data_labels_as_numbers = pd.concat([X_train, train_labels], axis=1)
val_data_labels_as_numbers = pd.concat([X_val, val_labels], axis=1)
test_data_labels_as_numbers = pd.concat([X_test, test_labels], axis=1)

train_data_labels_as_numbers.to_parquet("../01 - Inputs/train_data_labels_as_numbers.parquet", index=False)
val_data_labels_as_numbers.to_parquet("../01 - Inputs/val_data_labels_as_numbers.parquet", index=False)
test_data_labels_as_numbers.to_parquet("../01 - Inputs/test_data_labels_as_numbers.parquet", index=False)

In [14]:
print(train_data_labels_as_numbers.shape, val_data_labels_as_numbers.shape, test_data_labels_as_numbers.shape)
print(train_data.shape, val_data.shape, test_data.shape)

(28166, 2) (6036, 2) (6036, 2)
(28166, 2) (6036, 2) (6036, 2)
